## Final Workflow: ML based integration pipeline (Token Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd

amazon_books = pd.read_parquet(DATA_DIR / "amazon_df.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks_df.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads_df.parquet")

### Sample Datasets

In [3]:
amazon_books.shape, metabooks.shape, goodreads.shape

((271360, 12), (535159, 12), (52478, 12))

In [4]:
isbn_m2a = set(metabooks['isbn_clean']) & set(amazon_books['isbn_clean'])

In [5]:
isbn_m2g = set(metabooks['isbn_clean']) & set(goodreads['isbn_clean'])

In [6]:
metabooks_sample = metabooks[metabooks['isbn_clean'].isin(isbn_m2a)]
amazon_sample = amazon_books[amazon_books['isbn_clean'].isin(isbn_m2a)]
goodreads_sample = goodreads[goodreads['isbn_clean'].isin(isbn_m2g)]

In [7]:
az_sample_2 = amazon_books[~amazon_books['isbn_clean'].isin(isbn_m2g)].sample(30_000 - len(amazon_sample), random_state=42)
amazon_sample = pd.concat([amazon_sample, az_sample_2]).reset_index(drop=True)

In [8]:
mb_sample_2 = metabooks[~metabooks['isbn_clean'].isin(isbn_m2g)].sample(30_000 - len(metabooks_sample), random_state=42)
metabooks_sample = pd.concat([metabooks_sample, mb_sample_2]).reset_index(drop=True)

In [9]:
gr_sample_2 = goodreads[~goodreads['isbn_clean'].isin(isbn_m2g)].sample(30_000 - len(goodreads_sample), random_state=42)
goodreads_sample = pd.concat([goodreads_sample, gr_sample_2]).reset_index(drop=True)

In [10]:
metabooks_sample.shape, amazon_sample.shape, goodreads_sample.shape

((30000, 12), (30000, 12), (30000, 12))

### Entity Matching

In [11]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [12]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author_name',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author_name',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author_name',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower,
                     list_strategy='best_match'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

/Users/abd/Developer/team_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Token Blocker

In [13]:
from PyDI.entitymatching import TokenBlocker


blocker_m2a = TokenBlocker(
    metabooks_sample, amazon_sample,
    column='title',
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=3,
    ngram_type='word'
)

blocker_m2g = TokenBlocker(
    metabooks_sample, goodreads_sample,
    column='title',
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=2,
    ngram_type='word'
)

token_candidates_m2a = blocker_m2a.materialize()
token_candidates_m2g = blocker_m2g.materialize()

### Evaluate Blocking

In [14]:
from PyDI.entitymatching import EntityMatchingEvaluator


# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=token_candidates_m2a,
    blocker=blocker_m2a,
    test_pairs=test_m2a,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)

{'pair_completeness': 0.6585365853658537,
 'pair_quality': 0.00021090784108484745,
 'reduction_ratio': 0.9998577577777777,
 'total_candidates': 128018,
 'total_possible_pairs': 900000000,
 'true_positives_found': 27,
 'total_true_pairs': 41,
 'evaluation_timestamp': '2025-11-10T12:55:21.532261',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [15]:
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=token_candidates_m2g,
    blocker=blocker_m2g,
    test_pairs=test_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2g)

{'pair_completeness': 0.2727272727272727,
 'pair_quality': 7.1249641525241075e-06,
 'reduction_ratio': 0.9981286455555556,
 'total_candidates': 1684219,
 'total_possible_pairs': 900000000,
 'true_positives_found': 12,
 'total_true_pairs': 44,
 'evaluation_timestamp': '2025-11-10T12:55:45.286219',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

### ML-based Matcher

In [16]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor = FeatureExtractor(comparators)

train_features_m2a = feature_extractor.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [17]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [18]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [19]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a_tbsample.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g_tbsample.csv", index=False)

### Evaluate Matching

In [20]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a,
    out_dir=debug_output_dir
)

display(eval_results_m2a)

{'precision': 1.0,
 'recall': 0.6585365853658537,
 'f1': 0.7941176470588235,
 'accuracy': 0.93,
 'true_positives': 27,
 'false_positives': 0,
 'false_negatives': 14,
 'true_negatives': 159,
 'threshold_used': 0.0,
 'total_correspondences': 20861,
 'filtered_correspondences': 20861,
 'evaluation_timestamp': '2025-11-10T16:29:56.979681',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [21]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.9230769230769231,
 'recall': 0.2727272727272727,
 'f1': 0.4210526315789473,
 'accuracy': 0.835,
 'true_positives': 12,
 'false_positives': 1,
 'false_negatives': 32,
 'true_negatives': 155,
 'threshold_used': 0.0,
 'total_correspondences': 1355,
 'filtered_correspondences': 1355,
 'evaluation_timestamp': '2025-11-10T16:30:08.229183',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

### Cluster Analysis

In [22]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,20719,99.663283
1,3,68,0.327096
2,4,2,0.009620


In [23]:
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,1313,98.425787
1,3,21,1.574213


### Data Fusion

In [35]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a_tbsample.csv")
correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g_tbsample.csv")

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

22216

In [36]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author_name', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [37]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_tb_sample.jsonl")

fused_ml_tblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_tblocker):,}')
display(fused_ml_tblocker.head())

Fused rows: 21,052


,_id,_fusion_group_id,_fusion_sources,genres,id,publisher,price,page_count,isbn_clean,language,title,author_name,publish_year,num_rating,rating,_fusion_confidence,_fusion_metadata
0,amazon_218052,group_0,"[amazon_books, metabooks]","[['Romance', 'Contemporary']]",amazon_218052,Silhouette,1.35,192.0,0373484348,English,daring moves marriage on demand,linda lael miller,2001.0,309,4.1,0.818182,"{'genres_rule': 'union', 'genres_sources': ['m..."
1,amazon_30116,group_1,"[amazon_books, metabooks]","[['Literature', 'Fiction', 'Genre Fiction']]",amazon_30116,Del Rey Books,NaN,656.0,0345405668,English,american empire blood and iron,harry turtledove,2002.0,215,4.4,0.733333,"{'genres_rule': 'union', 'genres_sources': ['m..."
2,amazon_9650,group_2,"[amazon_books, metabooks]","[['Literature', 'Fiction', 'History', 'Critici...",amazon_9650,Carroll &amp; Graf Publishers,1.30,532.0,0786702141,English,the mammoth book of historical detectives mamm...,michael ashley,1995.0,23,4.2,0.831995,"{'genres_rule': 'union', 'genres_sources': ['m..."
3,amazon_73047,group_3,"[amazon_books, metabooks]","[[""Children's Books"", 'Literature', 'Fiction']]",amazon_73047,Penguin USA (J),5.80,112.0,0723261423,English,peter rabbits natural foods cookbook,arnold dobrin,1977.0,35,4.8,0.818182,"{'genres_rule': 'union', 'genres_sources': ['m..."
4,amazon_106912,group_4,"[amazon_books, metabooks]","[['Comics', 'Graphic Novels', 'Comic Strips']]",amazon_106912,Viking,36.50,NaN,0670806773,English,the new yorker cartoon album 19751985,the new yorker,1985.0,2,3.0,0.751316,"{'genres_rule': 'union', 'genres_sources': ['m..."


In [38]:
fused_ml_tblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21052 entries, 0 to 21051
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 21052 non-null  object 
 1   _fusion_group_id    21052 non-null  object 
 2   _fusion_sources     21052 non-null  object 
 3   genres              20088 non-null  object 
 4   id                  21052 non-null  object 
 5   publisher           21052 non-null  object 
 6   price               19304 non-null  float64
 7   page_count          19060 non-null  float64
 8   isbn_clean          21052 non-null  object 
 9   language            20974 non-null  object 
 10  title               21052 non-null  object 
 11  author_name         21052 non-null  object 
 12  publish_year        21050 non-null  float64
 13  num_rating          21052 non-null  int64  
 14  rating              21052 non-null  float64
 15  _fusion_confidence  21052 non-null  float64
 16  _fus

In [39]:
fused_ml_tblocker.to_parquet(PIPELINE_DIR / "fused_ml_tbsample.parquet")

### Fusion Evaluation

In [40]:
fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_tbsample.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")
golden_fused_dataset = golden_fused_dataset.rename(columns={"author": "author_name"})

In [44]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd

def _is_missing_value(v):
    if v is None:
        return True
    if isinstance(v, float) and np.isnan(v):
        return True
    if isinstance(v, (list, tuple, set, np.ndarray)):
        return len(v) == 0
    if isinstance(v, str):
        s = v.strip().lower()
        return s in ("", "nan", "none")
    return False

def array_set_equality_match(fused_value, gold_value) -> bool:
    """Case-insensitive set equality for strings, arrays, or stringified lists."""
    if _is_missing_value(fused_value) and _is_missing_value(gold_value):
        return True
    if _is_missing_value(fused_value) or _is_missing_value(gold_value):
        return False

    def to_clean_set(value):
        # unwrap 0-d or 1-element numpy arrays
        if isinstance(value, np.ndarray):
            # flatten and collapse 1-element arrays
            flat = value.flatten().tolist()
            if len(flat) == 1:
                value = flat[0]
            else:
                value = flat

        # parse stringified lists like "['Fiction','Drama']"
        if isinstance(value, str):
            s = value.strip()
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    value = parsed
                else:
                    # simple delimited string
                    value = re.split(r"[|,;/]", s)
            except Exception:
                # not parsable, split manually
                value = re.split(r"[|,;/]", s)

        # flatten iterable containers
        if isinstance(value, np.ndarray):
            value = value.tolist()
        if isinstance(value, (list, tuple, set)):
            items = []
            for v in value:
                # recursively parse if element looks like "['a','b']"
                if isinstance(v, str) and v.strip().startswith("["):
                    try:
                        inner = ast.literal_eval(v)
                        if isinstance(inner, (list, tuple, set)):
                            items.extend(inner)
                            continue
                    except Exception:
                        pass
                items.append(v)
            return {str(x).strip().lower() for x in items if not _is_missing_value(x)}

        # scalar fallback
        return {str(value).strip().lower()}

    fused_set = to_clean_set(fused_value)
    gold_set  = to_clean_set(gold_value)

    if not fused_set and not gold_set:
        return True
    return fused_set == gold_set



strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author_name", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("rating", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("genres", array_set_equality_match)

In [45]:
dupe_rows = fused_dataset[fused_dataset['isbn_clean'].duplicated(keep=False)]
dupe_rows.sort_values('isbn_clean')

,_id,_fusion_group_id,_fusion_sources,genres,id,publisher,price,page_count,isbn_clean,language,title,author_name,publish_year,num_rating,rating,_fusion_confidence,_fusion_metadata


In [46]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval_tb.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.455
  macro_accuracy: 0.456
  num_evaluated_records: 38828
  num_evaluated_attributes: 8
  total_evaluations: 303319
  total_correct: 137865
  genres_accuracy: 0.446
  genres_count: 37448
  price_accuracy: 0.529
  price_count: 36440
  publisher_accuracy: 0.238
  publisher_count: 38828
  page_count_accuracy: 0.514
  page_count_count: 35411
  author_name_accuracy: 0.536
  author_name_count: 38828
  title_accuracy: 0.351
  title_count: 38828
  publish_year_accuracy: 0.525
  publish_year_count: 38708
  rating_accuracy: 0.507
  rating_count: 38828

Overall Accuracy: 45.5%
